# Multi-Frequency Training Deep Dive

This notebook explores multi-frequency training in detail, including:
- How frequency levels work
- Impact on training dynamics
- Hyperparameter tuning
- Best practices

## Theory

In Nested Learning, parameters are grouped into **frequency levels**:

$$
\theta = \{\theta_{\text{fast}}, \theta_{\text{medium}}, \theta_{\text{slow}}\}
$$

Each level updates at its own frequency:
- $\theta_{\text{fast}}$: Every step (f=1)
- $\theta_{\text{medium}}$: Every $f_m$ steps (e.g., f=10)
- $\theta_{\text{slow}}$: Every $f_s$ steps (e.g., f=100)

This creates a **hierarchy of timescales** for learning.

In [ ]:
# Setup
import sys
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.models import NestedMLP
from src.optimizers import NestedOptimizerBuilder
from src.training import NestedTrainer

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Comparing Different Frequency Configurations

Let's compare different frequency settings on the same task.

In [ ]:
# Generate dataset
def generate_spiral_data(n_samples=1000, noise=0.1):
    """Generate 2D spiral classification dataset."""
    n_per_class = n_samples // 3
    
    X = []
    y = []
    
    for class_idx in range(3):
        r = np.linspace(0.1, 1, n_per_class)
        t = np.linspace(class_idx * 4, (class_idx + 1) * 4, n_per_class) + np.random.randn(n_per_class) * noise
        
        x = r * np.sin(t * 2.5)
        y_coord = r * np.cos(t * 2.5)
        
        X.append(np.stack([x, y_coord], axis=1))
        y.extend([class_idx] * n_per_class)
    
    X = np.vstack(X)
    y = np.array(y)
    
    # Add extra features
    X_extra = np.random.randn(len(X), 8) * 0.1
    X = np.hstack([X, X_extra])
    
    # Shuffle
    indices = np.random.permutation(len(X))
    X = X[indices]
    y = y[indices]
    
    return torch.FloatTensor(X), torch.LongTensor(y)

# Generate data
X_train, y_train = generate_spiral_data(1000)
X_test, y_test = generate_spiral_data(300)

# Visualize
plt.figure(figsize=(8, 6))
for i in range(3):
    mask = y_train == i
    plt.scatter(X_train[mask, 0], X_train[mask, 1], label=f'Class {i}', alpha=0.6)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Spiral Dataset')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Create dataloaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Configuration 1: Fast Adaptation [1, 5, 25]

In [ ]:
# Fast adaptation config
model1 = NestedMLP(10, [64, 32], 3).to(device)
builder1 = NestedOptimizerBuilder(model1, num_levels=3)
builder1.auto_assign_params('uniform')
opt1 = builder1.build(
    optimizer_types=['adam', 'sgd', 'sgd'],
    learning_rates=[0.01, 0.05, 0.1],
    frequencies=[1, 5, 25]
)

trainer1 = NestedTrainer(model1, opt1, nn.CrossEntropyLoss(), device, use_nested_optimizer=True)
history1 = trainer1.fit(train_loader, test_loader, epochs=50)

print(f"Config 1 [1, 5, 25] - Final test acc: {history1['val_accuracy'][-1]:.4f}")

### Configuration 2: Balanced [1, 10, 100]

In [ ]:
# Balanced config
model2 = NestedMLP(10, [64, 32], 3).to(device)
builder2 = NestedOptimizerBuilder(model2, num_levels=3)
builder2.auto_assign_params('uniform')
opt2 = builder2.build(
    optimizer_types=['adam', 'sgd', 'sgd'],
    learning_rates=[0.01, 0.05, 0.1],
    frequencies=[1, 10, 100]
)

trainer2 = NestedTrainer(model2, opt2, nn.CrossEntropyLoss(), device, use_nested_optimizer=True)
history2 = trainer2.fit(train_loader, test_loader, epochs=50)

print(f"Config 2 [1, 10, 100] - Final test acc: {history2['val_accuracy'][-1]:.4f}")

### Configuration 3: Slow Adaptation [1, 20, 200]

In [ ]:
# Slow adaptation config
model3 = NestedMLP(10, [64, 32], 3).to(device)
builder3 = NestedOptimizerBuilder(model3, num_levels=3)
builder3.auto_assign_params('uniform')
opt3 = builder3.build(
    optimizer_types=['adam', 'sgd', 'sgd'],
    learning_rates=[0.01, 0.05, 0.1],
    frequencies=[1, 20, 200]
)

trainer3 = NestedTrainer(model3, opt3, nn.CrossEntropyLoss(), device, use_nested_optimizer=True)
history3 = trainer3.fit(train_loader, test_loader, epochs=50)

print(f"Config 3 [1, 20, 200] - Final test acc: {history3['val_accuracy'][-1]:.4f}")

### Compare Configurations

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
axes[0].plot(history1['train_loss'], label='Fast [1,5,25]', linewidth=2)
axes[0].plot(history2['train_loss'], label='Balanced [1,10,100]', linewidth=2)
axes[0].plot(history3['train_loss'], label='Slow [1,20,200]', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss by Configuration')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test accuracy
axes[1].plot(history1['val_accuracy'], label='Fast [1,5,25]', linewidth=2)
axes[1].plot(history2['val_accuracy'], label='Balanced [1,10,100]', linewidth=2)
axes[1].plot(history3['val_accuracy'], label='Slow [1,20,200]', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Test Accuracy by Configuration')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
print("\nFinal Results:")
print(f"Fast [1,5,25]:        Acc = {history1['val_accuracy'][-1]:.4f}, Loss = {history1['train_loss'][-1]:.4f}")
print(f"Balanced [1,10,100]:  Acc = {history2['val_accuracy'][-1]:.4f}, Loss = {history2['train_loss'][-1]:.4f}")
print(f"Slow [1,20,200]:      Acc = {history3['val_accuracy'][-1]:.4f}, Loss = {history3['train_loss'][-1]:.4f}")

## Part 2: Impact of Learning Rates per Level

Learning rates can be different for each frequency level.

In [ ]:
# Test different LR configurations
lr_configs = [
    ([0.001, 0.01, 0.1], "Increasing"),
    ([0.01, 0.01, 0.01], "Uniform"),
    ([0.1, 0.01, 0.001], "Decreasing")
]

lr_histories = []

for lrs, name in lr_configs:
    print(f"\nTraining with {name} LR: {lrs}")
    
    model = NestedMLP(10, [64, 32], 3).to(device)
    builder = NestedOptimizerBuilder(model, num_levels=3)
    builder.auto_assign_params('uniform')
    opt = builder.build(
        optimizer_types=['adam', 'adam', 'adam'],
        learning_rates=lrs,
        frequencies=[1, 10, 100]
    )
    
    trainer = NestedTrainer(model, opt, nn.CrossEntropyLoss(), device, use_nested_optimizer=True)
    history = trainer.fit(train_loader, test_loader, epochs=30)
    lr_histories.append((history, name))
    
    print(f"  Final test acc: {history['val_accuracy'][-1]:.4f}")

In [ ]:
# Plot LR comparison
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for history, name in lr_histories:
    plt.plot(history['train_loss'], label=name, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss by LR Configuration')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for history, name in lr_histories:
    plt.plot(history['val_accuracy'], label=name, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.title('Test Accuracy by LR Configuration')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 3: Optimizer Types per Level

Different optimizers can be used for different frequency levels.

In [ ]:
# Test different optimizer combinations
opt_configs = [
    (['adam', 'adam', 'adam'], "All Adam"),
    (['adam', 'sgd', 'sgd'], "Adam/SGD Mix"),
    (['sgd', 'sgd', 'sgd'], "All SGD")
]

opt_histories = []

for opts, name in opt_configs:
    print(f"\nTraining with {name}: {opts}")
    
    model = NestedMLP(10, [64, 32], 3).to(device)
    builder = NestedOptimizerBuilder(model, num_levels=3)
    builder.auto_assign_params('uniform')
    opt = builder.build(
        optimizer_types=opts,
        learning_rates=[0.01, 0.01, 0.01],
        frequencies=[1, 10, 100],
        optimizer_kwargs=[{}, {'momentum': 0.9}, {'momentum': 0.9}]
    )
    
    trainer = NestedTrainer(model, opt, nn.CrossEntropyLoss(), device, use_nested_optimizer=True)
    history = trainer.fit(train_loader, test_loader, epochs=30)
    opt_histories.append((history, name))
    
    print(f"  Final test acc: {history['val_accuracy'][-1]:.4f}")

In [ ]:
# Plot optimizer comparison
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for history, name in opt_histories:
    plt.plot(history['train_loss'], label=name, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss by Optimizer Configuration')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for history, name in opt_histories:
    plt.plot(history['val_accuracy'], label=name, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.title('Test Accuracy by Optimizer Configuration')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Best Practices

Based on experiments and theory:

### 1. Frequency Selection
- **Default**: `[1, 10, 100]` works well for most tasks
- **Fast adaptation**: `[1, 5, 25]` for quickly changing tasks
- **Stable learning**: `[1, 20, 200]` for long training runs

### 2. Learning Rates
- **Increasing**: Fast params use lower LR, slow params use higher LR
  - Fast params update frequently → small steps
  - Slow params update rarely → larger steps
- Example: `[0.001, 0.01, 0.1]`

### 3. Optimizer Types
- **Adam for fast**: Good for quick adaptation
- **SGD with momentum for slow**: More stable for infrequent updates
- Example: `['adam', 'sgd', 'sgd']`

### 4. Parameter Assignment
- **Output layer → Fast**: Quick task adaptation
- **Hidden layers → Medium**: General features
- **Input layer → Slow**: Stable feature extraction

## Summary

Multi-frequency training provides flexibility in:
- Update frequencies for different parameters
- Learning rates per level
- Optimizer types per level
- Parameter assignment strategies

**Next**: Notebook 03 explores the Continuum Memory System for preventing catastrophic forgetting.